In [1]:
# Author: Gergely Zahoranszky-Kohalmi, PhD
#
# Email: gergely.zahoranszky-kohalmi@nih.gov
#
# Organization: National Center for Advancing Translational Sciences
#

In [2]:
#Env: routesim
import json
import pandas as pd
import rxnutils

from rxnutils import routes
from rxnutils.routes import readers
from rxnutils.routes.readers import read_reactions_dataframe
from rxnutils.routes import comparison
from rxnutils.routes.scoring import badowski_route_score

import os


/Users/zahoranszkykog2/anaconda3/envs/routesim/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DIR_IN = '../data/output/yield_compare/rxns/'

FNAME_IN_ALL =[]






FNAME_OUT = '../data/output/yield_comparison_result.tsv'



# This is a unit cost, i.e. we rely purely on the yield-based score for the alternative aggregated yield computation
RXN_UNIT_COST = 1.0

MOL_UNIT_COST = 1.0


In [4]:
# Functions

def compute_alternative_aggr_yield (route, rxn_avg_yield):
    
    return (badowski_route_score (route, mol_costs = None, average_yield = rxn_avg_yield, reaction_cost = RXN_UNIT_COST))



In [5]:

search_obj = os.scandir(DIR_IN)

for dir_item in search_obj:
    if dir_item.is_file():
        
        FNAME_IN_ALL.append(DIR_IN + dir_item.name)

print(FNAME_IN_ALL)

['../data/output/yield_compare/rxns/CFRDJNOBCVXNRR-UHFFFAOYSA-N_evidence_based_route_rxn.tsv', '../data/output/yield_compare/rxns/CFRDJNOBCVXNRR-UHFFFAOYSA-N_evidence_based_route_rxn mod.tsv']


In [6]:
unique_route_indices = []

# We need to process syhnthesis routes separately, due to the the way reaction_utils handles synthesis route metadata

all_route_index = []
all_alternative_aggregated_yield = []
all_fnames = []

for fname in FNAME_IN_ALL:

    try:
        df = pd.read_csv (fname, sep = '\t')

        unique_route_indices = list(set(list(df['route_index'])))

        for route_index in unique_route_indices:

            df_tmp = df[df['route_index'] == route_index].copy()

            routes = read_reactions_dataframe(df_tmp, smiles_column = 'rxsmiles', group_by = ['route_index', 'tm_inchikey'], metadata_columns = ['yield', 'rxid', 'boya_aggregated_yield'])


            R = list(routes)[0]

            print ('[*] Starting alternative aggregated yield computation ...')
            
            # Computing avg yield of reactions as required by the alternative aggr. yield computation method:
            df_avg_yields = df_tmp.groupby(['route_index'], as_index = False).agg({
                                                                    'yield': 'mean'
                                                                }).rename (columns = {'yield': 'avg_yield'})

            avg_yield = list(df_avg_yields['avg_yield'])[0]

            print (f'[*] Avg yield of reactions for route index {route_index}: {avg_yield} .')

            alt_aggr_yield = compute_alternative_aggr_yield (R, avg_yield)
            
            print (f'[*] Aggregated yield for route of index {route_index}: {alt_aggr_yield}')
            
            all_route_index.append (route_index)
            all_alternative_aggregated_yield.append (alt_aggr_yield)
            all_fnames.append(fname)



        df_yield_alt = pd.DataFrame ({
            'route_index': all_route_index,
            'alt_aggregated_yield': all_alternative_aggregated_yield,
            'input': all_fnames
        })

        print (df_yield_alt)

    except:
        print (f'[W] No route could have been reconstructed in file: {fname} .')



df_yield = df.groupby(['tm_inchikey', 'route_index'], as_index = False).first()

df_yield = df_yield.merge (df_yield_alt, on = 'route_index', how = 'inner')

print (df_yield)

[*] Starting alternative aggregated yield computation ...
[*] Avg yield of reactions for route index 1: 66.25 .
[*] Aggregated yield for route of index 1: 1.0304236560483448
[*] Starting alternative aggregated yield computation ...
[*] Avg yield of reactions for route index 2: 69.02799999999999 .
[*] Aggregated yield for route of index 2: 1.029189964305826
[*] Starting alternative aggregated yield computation ...
[*] Avg yield of reactions for route index 3: 68.912 .
[*] Aggregated yield for route of index 3: 1.0292393870015044
[*] Starting alternative aggregated yield computation ...
[*] Avg yield of reactions for route index 4: 69.45599999999999 .
[*] Aggregated yield for route of index 4: 1.0290086403511554
   route_index  alt_aggregated_yield  \
0            1              1.030424   
1            2              1.029190   
2            3              1.029239   
3            4              1.029009   

                                               input  
0  ../data/output/yield_

In [7]:


print (df_avg_yields)

   route_index  avg_yield
0          199     69.456


In [8]:


print (df_yield_alt)

df_yield_alt.to_csv (FNAME_OUT, sep = '\t', index = False)

print ('[Done.]')

   route_index  alt_aggregated_yield  \
0            1              1.030424   
1            2              1.029190   
2            3              1.029239   
3            4              1.029009   
4            1              1.030424   
5            2              1.029190   
6            3              1.029239   
7            4              1.029009   
8          199              1.029009   

                                               input  
0  ../data/output/yield_compare/rxns/CFRDJNOBCVXN...  
1  ../data/output/yield_compare/rxns/CFRDJNOBCVXN...  
2  ../data/output/yield_compare/rxns/CFRDJNOBCVXN...  
3  ../data/output/yield_compare/rxns/CFRDJNOBCVXN...  
4  ../data/output/yield_compare/rxns/CFRDJNOBCVXN...  
5  ../data/output/yield_compare/rxns/CFRDJNOBCVXN...  
6  ../data/output/yield_compare/rxns/CFRDJNOBCVXN...  
7  ../data/output/yield_compare/rxns/CFRDJNOBCVXN...  
8  ../data/output/yield_compare/rxns/CFRDJNOBCVXN...  
[Done.]


In [9]:
# References:
#
# Ref: https://stackoverflow.com/questions/20199126/reading-json-from-a-file
# Ref: https://stackoverflow.com/questions/12943819/how-to-prettyprint-a-json-file
# Ref: https://stackoverflow.com/questions/32468402/how-to-explode-a-list-inside-a-dataframe-cell-into-separate-rows
# Ref: Google AI Overview, 04/30/2026, by Google Gemini AI built-in Chrome
# Ref: https://www.geeksforgeeks.org/python/python-list-files-in-a-directory/
# Ref: 
#



